# Workflows de datos con LangGraph
Organizaremos un análisis del catálogo de documentos mediante estado, nodos y rutas.
El flujo será determinista: sus decisiones se programan en Python, sin llamadas a un modelo.

## Contenido
1. Estado y nodos.
2. Construcción y ejecución.
3. Rutas condicionales y seguimiento.
4. Subgraphs y sus interfaces.
5. Dos ejercicios.


## Del proceso al estado
Antes de elegir métodos de LangGraph, describe el trabajo que quieres automatizar:
recibir un tema, comprobar la entrada, seleccionar documentos y producir un resumen.
Después identifica las operaciones, los datos que necesita cada paso y las decisiones de ruta.

| Paso | Lee | Produce |
|---|---|---|
| Normalizar | Tema original | Tema normalizado |
| Seleccionar | Tema y catálogo | Documentos encontrados |
| Resumir | Documentos | Conteo, páginas y mensaje |
| Informar ausencia | Tema | Mensaje sin resultados |

Conservaremos el tema original para compararlo con el normalizado.
Los documentos siguen siendo registros; el mensaje se construye al final como salida para el usuario.
Este diseño permite sustituir un paso de búsqueda sin rehacer el resto del flujo.
Seguimos el enfoque de [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph), aplicado a nuestro catálogo.


## 1. Estado y nodos
En Neo4j, un nodo representa una entidad almacenada y una relación describe un vínculo entre entidades.
En LangGraph, un nodo ejecuta una función y una arista define qué paso sigue.
El **estado** contiene los datos que esos pasos comparten.

Aquí el catálogo es el mismo de la sesión anterior, incluido en este proyecto para ejecutarlo de forma independiente.


In [1]:
from sample_data import DOCUMENTS

for document in DOCUMENTS:
    print(document)


{'title': 'Python fundamentals', 'topic': 'python', 'pages': 28}
{'title': 'Text embeddings', 'topic': 'retrieval', 'pages': 12}
{'title': 'MCP tools', 'topic': 'integration', 'pages': 8}
{'title': 'Graph search', 'topic': 'retrieval', 'pages': 16}
{'title': 'NumPy arrays', 'topic': 'data', 'pages': 20}
{'title': 'Pandas tables', 'topic': 'data', 'pages': 24}


### Definir el estado
`TypedDict` describe las claves y sus tipos; no convierte ni valida datos durante la ejecución.
`total=False` permite que las claves aparezcan conforme avanza el flujo: al comenzar solo entregaremos `topic`.
Un nodo devuelve las claves que actualiza; las demás conservan su valor.
En este ejemplo, una actualización reemplaza el valor anterior de esa clave.


In [2]:
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph
from sample_data import DOCUMENTS, Document


class WorkflowState(TypedDict, total=False):
    topic: str
    original_topic: str
    documents: list[Document]
    count: int
    total_pages: int
    message: str


### Un nodo es una función
El primer paso normaliza el tema. Recibe el estado y devuelve una actualización, sin modificar el diccionario original.


In [3]:
def normalize_topic(state: WorkflowState) -> WorkflowState:
    return {"original_topic": state["topic"], "topic": state["topic"].strip().lower()}


In [4]:
input_state: WorkflowState = {"topic": " Retrieval "}
print(normalize_topic(input_state))
print(input_state)


{'original_topic': ' Retrieval ', 'topic': 'retrieval'}
{'topic': ' Retrieval '}


## 2. Construcción y ejecución
`StateGraph` recibe el esquema. Registramos la función con un nombre y conectamos `START` y `END`, que marcan entrada y salida.
`compile()` prepara el grafo para ejecutarlo; `invoke()` devuelve el estado final.


In [5]:
builder = StateGraph(WorkflowState)
builder.add_node("normalize", normalize_topic)
builder.add_edge(START, "normalize")
builder.add_edge("normalize", END)
simple_workflow = builder.compile()

print(simple_workflow.invoke({"topic": " Retrieval "}))


{'topic': 'retrieval', 'original_topic': ' Retrieval '}


## 3. Rutas condicionales
Ampliemos el flujo: normalizar → seleccionar → resumir o informar que no hubo resultados.
La función de selección filtra el catálogo. Los nodos posteriores podrán consultar `documents`.


In [6]:
def select_documents(state: WorkflowState) -> WorkflowState:
    selected = []
    for document in DOCUMENTS:
        if document["topic"] == state["topic"]:
            selected.append(document)
    return {"documents": selected}


### Elegir el siguiente paso
La función de ruta devuelve una etiqueta. `add_conditional_edges()` relacionará cada etiqueta con el nombre de un nodo.
Esta función decide el destino; no produce una actualización del estado.


In [7]:
def route_documents(state: WorkflowState) -> Literal["found", "empty"]:
    if state["documents"]:
        return "found"
    return "empty"


### Dos salidas posibles
Con documentos, contamos filas y páginas. Sin documentos, producimos un mensaje y valores cero.


In [8]:
def summarize_documents(state: WorkflowState) -> WorkflowState:
    count = len(state["documents"])
    total_pages = sum(document["pages"] for document in state["documents"])
    return {
        "count": count,
        "total_pages": total_pages,
        "message": f"Found {count} documents with {total_pages} pages.",
    }


def report_empty(state: WorkflowState) -> WorkflowState:
    return {
        "count": 0,
        "total_pages": 0,
        "message": f"No documents found for topic: {state['topic']}",
    }


### Conectar los componentes
Separamos la construcción en una función para poder crear un grafo nuevo cuando cambien los nodos o las conexiones.
Las rutas `found` y `empty` terminan en nodos distintos, y ambas llegan a `END`.


In [9]:
def build_workflow():
    builder = StateGraph(WorkflowState)
    builder.add_node("normalize", normalize_topic)
    builder.add_node("select", select_documents)
    builder.add_node("summarize", summarize_documents)
    builder.add_node("empty", report_empty)
    builder.add_edge(START, "normalize")
    builder.add_edge("normalize", "select")
    builder.add_conditional_edges(
        "select", route_documents,
        {"found": "summarize", "empty": "empty"},
    )
    builder.add_edge("summarize", END)
    builder.add_edge("empty", END)
    return builder.compile()


In [10]:
workflow = build_workflow()
result = workflow.invoke({"topic": " Retrieval "})
print(result["message"])
print([document["title"] for document in result["documents"]])

empty_result = workflow.invoke({"topic": "unknown"})
print(empty_result["message"])
assert result["count"] == 2
assert result["total_pages"] == 28
assert empty_result["count"] == 0


Found 2 documents with 28 pages.
['Text embeddings', 'Graph search']
No documents found for topic: unknown


### Observar los pasos
`stream(..., stream_mode="updates")` ejecuta el flujo y entrega las actualizaciones de cada nodo, en lugar de esperar solamente al resultado final.
No es streaming de tokens de un modelo: aquí estamos viendo cambios del estado.


In [11]:
for update in workflow.stream({"topic": "data"}, stream_mode="updates"):
    print(update)


{'normalize': {'original_topic': 'data', 'topic': 'data'}}
{'select': {'documents': [{'title': 'NumPy arrays', 'topic': 'data', 'pages': 20}, {'title': 'Pandas tables', 'topic': 'data', 'pages': 24}]}}
{'summarize': {'count': 2, 'total_pages': 44, 'message': 'Found 2 documents with 44 pages.'}}


## 4. Subgraphs y sus interfaces
Un **subgraph** reúne varios pasos en un workflow compilado que otro grafo puede reutilizar como nodo.
Agruparemos selección y sus dos salidas en `analysis`; el padre se encargará de validar y normalizar la entrada.
No cambia el análisis del catálogo: cambia cómo organizamos los componentes.

### Validar la entrada antes de analizar
Una búsqueda válida sin coincidencias toma la ruta `empty`.
Una entrada vacía o de un tipo incorrecto es un problema de la petición: la rechazamos antes de buscar.
`TypedDict` no realiza esa validación por nosotros.


In [12]:
def validate_request(state: WorkflowState) -> WorkflowState:
    topic = state.get("topic")
    if not isinstance(topic, str):
        raise TypeError("topic must be a string")
    if not topic.strip():
        raise ValueError("topic must not be blank")
    return {}


In [13]:
for topic in ("retrieval", "   "):
    try:
        validate_request({"topic": topic})
        print("Accepted:", repr(topic))
    except ValueError as error:
        print("Rejected:", error)


Accepted: 'retrieval'
Rejected: topic must not be blank


### Compilar el subgraph
Reutilizamos las funciones anteriores. Este componente recibe `topic` y produce los resultados del análisis.
Podemos probarlo de forma independiente con un tema ya normalizado.


In [14]:
def build_analysis_subgraph():
    builder = StateGraph(WorkflowState)
    builder.add_node("select", select_documents)
    builder.add_node("summarize", summarize_documents)
    builder.add_node("empty", report_empty)
    builder.add_edge(START, "select")
    builder.add_conditional_edges(
        "select", route_documents,
        {"found": "summarize", "empty": "empty"},
    )
    builder.add_edge("summarize", END)
    builder.add_edge("empty", END)
    return builder.compile()


In [15]:
analysis = build_analysis_subgraph()
print(analysis.invoke({"topic": "retrieval"})["message"])


Found 2 documents with 28 pages.


### Integrarlo como nodo
El padre y el hijo comparten las claves de `WorkflowState`, por lo que el grafo compilado se puede pasar directamente a `add_node()`.
El recorrido del padre será validar → normalizar → analysis; dentro de `analysis` siguen existiendo las dos rutas.


In [16]:
def build_modular_workflow():
    analysis = build_analysis_subgraph()
    builder = StateGraph(WorkflowState)
    builder.add_node("validate", validate_request)
    builder.add_node("normalize", normalize_topic)
    builder.add_node("analysis", analysis)
    builder.add_edge(START, "validate")
    builder.add_edge("validate", "normalize")
    builder.add_edge("normalize", "analysis")
    builder.add_edge("analysis", END)
    return builder.compile()


In [17]:
modular_workflow = build_modular_workflow()
for topic in (" Retrieval ", "unknown"):
    result = modular_workflow.invoke({"topic": topic})
    print(result["original_topic"], "->", result["topic"])
    print(result["message"])

try:
    modular_workflow.invoke({"topic": "   "})
except ValueError as error:
    print("Rejected before analysis:", error)


 Retrieval  -> retrieval
Found 2 documents with 28 pages.
unknown -> unknown
No documents found for topic: unknown
Rejected before analysis: topic must not be blank


### Observar padre e hijo
`subgraphs=True` incluye actualizaciones internas del hijo.
Cada elemento contiene un namespace —vacío para el padre o identificado para el hijo— y la actualización del paso.


In [18]:
for namespace, update in modular_workflow.stream(
    {"topic": "data"}, stream_mode="updates", subgraphs=True
):
    print(namespace, update)


() {'validate': None}
() {'normalize': {'original_topic': 'data', 'topic': 'data'}}
('analysis:4b624f36-021e-f90f-b79c-1010f92561dc',) {'select': {'documents': [{'title': 'NumPy arrays', 'topic': 'data', 'pages': 20}, {'title': 'Pandas tables', 'topic': 'data', 'pages': 24}]}}
('analysis:4b624f36-021e-f90f-b79c-1010f92561dc',) {'summarize': {'count': 2, 'total_pages': 44, 'message': 'Found 2 documents with 44 pages.'}}
() {'analysis': {'topic': 'data', 'original_topic': 'data', 'documents': [{'title': 'NumPy arrays', 'topic': 'data', 'pages': 20}, {'title': 'Pandas tables', 'topic': 'data', 'pages': 24}], 'count': 2, 'total_pages': 44, 'message': 'Found 2 documents with 44 pages.'}}


### Cuando las claves son diferentes
Si el padre usa `query` y `report`, y el hijo usa `topic` y `message`, una función adapta la entrada y la salida.
No basta con registrar el subgraph directamente: hay que indicar la correspondencia entre las claves.


In [19]:
class RequestState(TypedDict, total=False):
    query: str
    report: str


analysis = build_analysis_subgraph()


def call_analysis(state: RequestState) -> RequestState:
    result = analysis.invoke({"topic": state["query"].strip().lower()})
    return {"report": result["message"]}


request_builder = StateGraph(RequestState)
request_builder.add_node("analyze", call_analysis)
request_builder.add_edge(START, "analyze")
request_builder.add_edge("analyze", END)
request_workflow = request_builder.compile()


In [20]:
result = request_workflow.invoke({"query": " Retrieval "})
print(result)
assert result["report"] == "Found 2 documents with 28 pages."


{'query': ' Retrieval ', 'report': 'Found 2 documents with 28 pages.'}


La separación permite reutilizar el mismo análisis desde dos interfaces.
No estamos creando otro agente ni añadiendo un modelo: un subgraph también puede contener únicamente funciones Python.
[Guía oficial de subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs).


## 5. Ejercicios
### Ejercicio 1 · Filtrar por páginas
Añade `min_pages: int` al estado y adapta `select_documents` para que respete ese mínimo además del tema.
Usa `state.get("min_pages", 0)` para conservar las llamadas previas que no indican un mínimo.
Construye otra vez el grafo y muestra dos ejecuciones con `topic="retrieval"`: mínimo 14 (un documento) y mínimo 20 (ninguno).
Conserva las dos rutas existentes.


In [21]:
# Write your answer here.


### Ejercicio 2 · Añadir un paso al flujo
Añade `average_pages: float` al estado y un nodo `calculate_average` que calcule la media de páginas de los documentos encontrados.
Conéctalo entre `summarize` y `END`; la ruta vacía debe seguir yendo directamente a `END`.
Muestra el resultado con `topic="data"` y `min_pages=0`, y explica qué cambió en las conexiones.


In [22]:
# Write your answer here.


## Referencias
- [LangGraph: overview](https://docs.langchain.com/oss/python/langgraph/overview).
- [Graph API: estado, nodos y aristas](https://docs.langchain.com/oss/python/langgraph/graph-api).
- [Workflows y agentes](https://docs.langchain.com/oss/python/langgraph/workflows-agents).
- [Streaming](https://docs.langchain.com/oss/python/langgraph/streaming).
- [TypedDict](https://docs.python.org/3/library/typing.html#typing.TypedDict).
